# AmEx Credit Card Product Assistant — Example Usage of `file_search` Library

This notebook demonstrates how to build a **realtime sales cue engine for AmEx credit cards** on top of the reusable `file_search` library.

**Use case:** During a customer call, a customer raises an objection or asks a question about an AmEx card → your existing system detects the objection and retrieves similar past objection-response pairs → this engine grounds the suggested response in authoritative AmEx product information (features, benefits, rewards, fees, APR, eligibility) to prevent hallucination.

**Why grounding matters:** Credit card products have regulated disclosures. Hallucinating an APR, a reward multiplier, or an annual fee has compliance implications, not just quality ones. This notebook demonstrates how to build a grounded assistant with automatic query decomposition and source-cited answers.

This serves as a **reference pattern** for other teams building RAG use cases on the library.

## 1. Setup — import the library

In [ ]:
import os
from file_search import VectorStore, QueryEngine

## 2. One-time indexing

Each file is parsed, word-window-chunked (600-word chunks, 300-word overlap), embedded, and stored. The library auto-derives the `source` field from the file stem — no other metadata is stored.

In [ ]:
INDEX_PATH = "amex_card_index.parquet"

if not os.path.exists(INDEX_PATH):
    store = VectorStore()

    store.add_file("amex_product_docs/platinum_card.pdf")
    store.add_file("amex_product_docs/gold_card.pdf")
    store.add_file("amex_product_docs/green_card.pdf")
    store.add_file("amex_product_docs/blue_cash_preferred.pdf")
    store.add_file("amex_product_docs/delta_skymiles.pdf")
    store.add_file("amex_product_docs/membership_rewards_terms.md")
    store.add_file("amex_product_docs/fees_and_apr_disclosures.pdf")

    store.save(INDEX_PATH)
else:
    print(f"Index already exists at {INDEX_PATH}")

## 3. Load the index at startup

In your realtime service, load once at startup and reuse across all queries.

In [ ]:
product_store = VectorStore.load(INDEX_PATH)
print(f"Loaded {len(product_store.chunks)} chunks from {len(set(product_store.sources))} sources")

print("\nSample chunk:")
print(product_store.chunks[0][:400])

## 4. Configure the QueryEngine for AmEx card cue

The system message tells the LLM to ground every fact in the provided CONTEXT and to cite sources via the `[Source: <filename>]` line that precedes each chunk. This is critical for credit card products where incorrect disclosures have compliance implications.

In [ ]:
AMEX_CARD_SYSTEM_MESSAGE = (
    "You are a real-time sales assistant helping representatives respond to customer "
    "questions and objections about American Express credit cards. Suggest a concise "
    "response (2-3 sentences) the rep can use.\n\n"
    "The CONTEXT below contains authoritative product facts — the official terms and "
    "disclosures. Do not contradict or supplement them.\n\n"
    "CRITICAL RULES:\n"
    "1. Each fact in CONTEXT is preceded by a `[Source: <filename>]` line identifying "
    "which product document it came from (e.g. `platinum_card`, `gold_card`, "
    "`fees_and_apr_disclosures`, `membership_rewards_terms`). Attribute each feature, "
    "fee, benefit, or term to the correct product using that source filename.\n"
    "2. If the customer asks about a specific card, use ONLY facts sourced from that "
    "card's document. Ignore facts sourced from other cards unless the customer "
    "explicitly asked for a comparison.\n"
    "3. When comparing cards, clearly label which feature belongs to which card. "
    "Example: 'The Platinum Card offers X, while the Gold Card offers Y.'\n"
    "4. All product details (fees, APRs, reward rates, benefits) MUST come from the "
    "CONTEXT. NEVER invent values — incorrect disclosures have compliance "
    "implications.\n"
    "5. If CONTEXT for the specific card asked about is missing, instruct the rep to "
    "confirm and follow up rather than guess.\n"
    "6. Cite sources inline as `[Source: <filename>]` whenever you state a specific "
    "fact from CONTEXT.\n"
    "7. Output ONLY the suggested response — no preamble."
)

engine = QueryEngine(
    product_store,
    system_message=AMEX_CARD_SYSTEM_MESSAGE,
)

## 5. Demo — large, messy customer query

In practice the input is messy — a full call transcript, a pasted email, a batch of utterances, a multi-topic question. You assemble whatever prompt you want from your own data and hand it to `retrieve()`, which splits it internally into sub-queries, retrieves per sub-query, and RRF-merges across sub-queries. Then `synthesize()` grounds the answer.

In [ ]:
# A deliberately messy, multi-topic customer input.
# In production this could be a call transcript, pasted email,
# concatenated chat history, etc. — whatever you've got.
messy_query = """
Hey so I was looking at the Platinum Card, $695 seems steep honestly,
what do I actually get for it, also my wife spends $1500/month on
groceries so is Gold or Blue Cash Preferred better for her, and does
the Delta SkyMiles card cover lounge access at DTW? One more thing,
what's the foreign transaction fee on Green? Sorry for the long message.
"""

# Hybrid retrieve (long queries are split internally and RRF-merged).
hits = product_store.retrieve(messy_query, top_k=10, final_k=5)
print(f"Retrieved {len(hits)} chunks (RRF-merged across auto-split sub-queries):")
for h in hits:
    print(f"  [score={h['score']:.4f}] {h['source']}")

# Synthesize a grounded answer using the retrieved chunks.
result = engine.synthesize(messy_query, hits)
print("\nANSWER:\n")
print(result["answer"])

## 6. Tuning knobs

| Knob | Default | Notes |
|---|---|---|
| `top_k` | `10` | Candidates per sub-query from hybrid search |
| `final_k` | `5` | Upper bound on chunks returned after RRF merge (may be less with heavy overlap; may exceed `top_k` when multiple sub-queries contribute unique chunks) |
| `CHUNK_SIZE` | `600` | Words per document chunk and per query sub-window (module constant in `file_search.py`) |
| `CHUNK_OVERLAP` | `300` | Word overlap between consecutive chunks |

All network calls (embeddings and chat completions) are wrapped in a universal burst-retry: within a burst, 5 rapid attempts (~0.2 s apart); between bursts, exponential backoff (2 / 4 / 8 / 16 s); up to 5 bursts total (25 max attempts per call). Retries trigger on **any** exception — transient network errors, rate limits, unexpected API responses all get the same treatment.